# Clustering Debug Embeddings Demo

This notebook parses successful attempts from `clustering_debug.jsonl`, selects one successful clustering prompt, reuses the local embedding cache for `text-embedding-3-small`, and shows a small diversity-sampled set of strings with their cluster assignments for that single prompt.

You can now truncate vectors after loading the cache by keeping only the first `d` dimensions before normalization and diversity sampling.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "clustering_debug_embeddings.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from clustering_debug_embeddings import (
    OPENAI_EMBEDDING_FULL_DIMENSIONS_BY_MODEL,
    extract_successful_cluster_attempts,
    run_sampling_pipeline,
    select_successful_attempt,
    summaries_for_attempt,
)

INPUT_PATH = Path(
    "/users/PAA0201/ollieproudman/work/DecomposedReasoning/Analysis/output/manual_20260410/"
    "aime25_qwen3_8b_checkpoint456_branching_cluster_across_bp010_b2_d5_32k_mc24/"
    "aime25_sft_branching_cluster_across_seed1234_20260412T021333.840131Z/"
    "clustering_debug.jsonl"
)
EMBEDDING_MODEL = "text-embedding-3-small"
SUCCESS_INDEX = 1
SAMPLE_COUNT = 5
BATCH_SIZE = 1024
EMBEDDING_DIMENSIONS = None  # Try 64, 128, 256, 512, 1024, or 1536.
FULL_EMBEDDING_DIMENSIONS = OPENAI_EMBEDDING_FULL_DIMENSIONS_BY_MODEL.get(
    EMBEDDING_MODEL,
)

INPUT_PATH, FULL_EMBEDDING_DIMENSIONS

(PosixPath('/users/PAA0201/ollieproudman/work/DecomposedReasoning/Analysis/output/manual_20260410/aime25_qwen3_8b_checkpoint456_branching_cluster_across_bp010_b2_d5_32k_mc24/aime25_sft_branching_cluster_across_seed1234_20260412T021333.840131Z/clustering_debug.jsonl'),
 PosixPath('/users/PAA0201/ollieproudman/work/DecomposedReasoning/Analysis/output/manual_20260410/aime25_qwen3_8b_checkpoint456_branching_cluster_across_bp010_b2_d5_32k_mc24/aime25_sft_branching_cluster_across_seed1234_20260412T021333.840131Z/clustering_debug.text-embedding-3-small.embeddings.json'))

In [2]:
if FULL_EMBEDDING_DIMENSIONS is None:
    print("Unknown default full embedding dimension for this model.")
else:
    print(f"Default full embedding dimension: {FULL_EMBEDDING_DIMENSIONS}")
print(f"Requested embedding dimensions: {EMBEDDING_DIMENSIONS}")

Detected full embedding dimension from cache: 1536
Requested embedding dimensions: None


In [3]:
attempts = extract_successful_cluster_attempts(input_path=INPUT_PATH)
selected_attempt = select_successful_attempt(
    attempts=attempts,
    success_index=SUCCESS_INDEX,
)
text_summaries = summaries_for_attempt(attempt=selected_attempt)

print(f"successful attempts: {len(attempts)}")
print(f"selected success_index: {selected_attempt.success_index}")
print(f"selected attempt_number: {selected_attempt.attempt_number}")
print(f"cleaned prompt items: {len(text_summaries)}")
print(f"actual clusters: {len(selected_attempt.groups)}")

successful attempts: 728
selected success_index: 1
selected attempt_number: 1
cleaned prompt items: 37
actual clusters: 5


In [4]:
run = run_sampling_pipeline(
    input_path=INPUT_PATH,
    success_index=SUCCESS_INDEX,
    requested_sample_count=SAMPLE_COUNT,
    embedding_model=EMBEDDING_MODEL,
    batch_size=BATCH_SIZE,
    embedding_dimensions=EMBEDDING_DIMENSIONS,
)

rows = run.display_rows()
rows

[{'text': 'Set up equation for integer quotient',
  'occurrences': 19,
  'attempts': 1,
  'clusters': 'Set up integer quotient equation [quot_eq] x19'},
 {'text': 'Identify valid base constraints',
  'occurrences': 1,
  'attempts': 1,
  'clusters': 'Identify integer solutions [int_sol] x1'},
 {'text': 'Identify integer solutions for k',
  'occurrences': 1,
  'attempts': 1,
  'clusters': 'Identify integer solutions [int_sol] x1'},
 {'text': 'Set remainder to zero',
  'occurrences': 3,
  'attempts': 1,
  'clusters': 'Set up divisibility condition [div_cond] x3'},
 {'text': 'Identify divisors of constant term',
  'occurrences': 1,
  'attempts': 1,
  'clusters': 'Set up divisibility condition [div_cond] x1'}]

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    pd.DataFrame(rows)
else:
    rows

In [ ]:
import time

DIMENSION_EXPERIMENTS = [64, 128, 256, 512, 1024, 1536]
comparison_rows = []

for embedding_dimensions in DIMENSION_EXPERIMENTS:
    starttime = time.time()
    if (
        FULL_EMBEDDING_DIMENSIONS is not None
        and embedding_dimensions > FULL_EMBEDDING_DIMENSIONS
    ):
        continue
    comparison_run = run_sampling_pipeline(
        input_path=INPUT_PATH,
        success_index=SUCCESS_INDEX,
        requested_sample_count=SAMPLE_COUNT,
        embedding_model=EMBEDDING_MODEL,
        batch_size=BATCH_SIZE,
        embedding_dimensions=embedding_dimensions,
    )
    endtime = time.time()
    comparison_rows.append(
        {
            "embedding_dimensions": embedding_dimensions,
            "sampled_texts": " | ".join(
                summary.text for summary in comparison_run.sampled_summaries
            ),
            "execution_time": endtime - starttime,
        }
    )

if pd is not None:
    pd.DataFrame(comparison_rows)
else:
    comparison_rows

In [ ]:
top_rows = [summary.to_display_row() for summary in run.text_summaries[:25]]

if pd is not None:
    pd.DataFrame(top_rows)
else:
    top_rows